# RNN Language Model — 실험 노트북

## 학습 목표
- PyTorch RNN으로 여러 데이터셋을 학습·평가하며 RNN LM의 **성능과 한계**를 확인
- 각 데이터셋을 "명제 하나를 증명하는 실험"으로 설계 (**baseline · metric** 포함)
- WandB로 학습 과정을 시각화하고 각 실험에 맞는 metric으로 평가
- **학습(teacher forcing) vs 추론(autoregressive)** 의 동작 차이 관찰

## 노트북 구성 원칙
각 실험은 **명제 → 데이터 → baseline → RNN → 결론**의 5단 리듬으로 완결된다.
목표는 "loss가 내려간다"가 아니라 **"고정 baseline이 못 하는 것을 RNN이 한다"** 를 보이는 것.

## 실험 설계 요약

| 실험 | 증명할 명제 | Metric | Baseline |
|---|---|---|---|
| **A. 반복 데이터** | RNN이 고정 윈도우보다 **긴 맥락을 활용**한다 | held-out next-token accuracy | bigram / n-gram (count) |
| **B. Long-range** | 간격 N이 커지면 **장기의존성이 무너진다** | recall 위치 accuracy (중간 마스킹) | n-gram(구조적 불가) + chance |
| **C. 실제 텍스트** | **학습 ≠ 추론** (exposure bias) | TF perplexity vs 자기회귀 생성 붕괴 | (정성 데모) |

> **학습 목표 매핑** — A·B: 성능·한계 정량 / C: 학습·추론 차이 / WandB: 전 실험 공통

## 1. 공통 코드베이스 구축
해당 실험에서 공유 가능한 코드베이스를 구축. 데이터 생성 및 평가는 각 실험 파트에서 구현하도록 함


### 1.1 Import
해당 노트북에서 필요한 Python Package를 Import 한다.

In [23]:
# 1. Python Standard Library
import re
import random
from typing import Dict, List, Set, Tuple, Callable

# 2. Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# 3. Tracking Tools
import wandb
from tqdm.auto import tqdm

### 1.2 RNN Language Model 정의
- `nn.Module`을 이용해 RNN Language Model을 정의
    - Model Input: Token들의 Embedding Vector
    - Model Output: Vocab 내 Token 각각의 예측값

![image.png](https://static.wikidocs.net/images/page/46496/rnnlm1_final_final.PNG)

In [24]:
class RNNLM(nn.Module):
    def __init__(
        self, vocab_size: int, hidden_size: int, output_size: int, *args, **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.rnn = nn.RNN(input_size=vocab_size, hidden_size=hidden_size)
        self.linear = nn.Linear(in_features=hidden_size, out_features=output_size)

    def forward(self, x: torch.FloatTensor, hidden = None) -> Tuple[torch.FloatTensor, torch.FloatTensor]:
        """
        Args:
            x (batch_size, seq_len, vocab_size): 입력 문자열 시퀀스의 One-hot vector
            hidden (hidden_size,): RNN의 초기 hidden state
        Return:
            y (batch_size, seq_len, vocab_size): 입력 시퀀스 내 각 위치에서 다음 토큰의 예측값
            hidden (batch_size, hidden_size): 입력 시퀀스의 마지막 위치에서의 hidden state
        """
        if x.ndim == 2:
            # 만약 (batch_size, ...)의 입력이 아닌, (...) 형태의 입력이 들어온다면,
            # batch_size 차원을 추가해주기 위해 아래와 같이 동작하도록 한다.
            x = x.unsqueeze(0)

        output, hidden = self.rnn(x, hidden)
        y = self.linear(output)
        return y, hidden

### 1.3 데이터 전처리 (Preprocessing)
- Tokenizer 정의
    - 문자 단위 Tokenizer, 단어 단위 Tokenizer 사용
    - 영어 소문자와 띄어쓰기, 기초적인 문장부호가 포함된 문장
- Vocabulary 구축
    - Tokenize 과정에서 발생한 토큰을 Vocabulary화
    - token2idx, idx2token을 가져 encode, decode를 수행할 수 있도록 함

In [25]:
def char_tokenize(text: str) -> List[str]:
    """
    문자 기반으로 토크나이징을 수행하는 함수
    
    Args:
        text (str): 토크나이징을 수행하고자 하는 텍스트
    Returns:
        List[str]: 문자 기반으로 쪼개진 토큰들이 담긴 리스트
    """
    return list(text)

def word_tokenize(text: str) -> List[str]:
    """
    단어 기반으로 토크나이징을 수행하는 함수

    Args:
        text (str): 토크나이징을 수행하고자 하는 텍스트
    Returns:
        List[str]: 단어 기반으로 쪼개진 토큰들이 담긴 리스트
    """
    return re.findall(r"[a-zA-Z']+|[.,!?]|\s+", text)

print(f"{char_tokenize('this is an apple.')=}")
print(f"{word_tokenize('this is an apple.')=}")

char_tokenize('this is an apple.')=['t', 'h', 'i', 's', ' ', 'i', 's', ' ', 'a', 'n', ' ', 'a', 'p', 'p', 'l', 'e', '.']
word_tokenize('this is an apple.')=['this', ' ', 'is', ' ', 'an', ' ', 'apple', '.']


In [26]:
def get_vocab(corpus: List[str], tokenize_fn: Callable) -> List[str]:
    """
    입력된 전체 corpus에 대해서 정렬된 vocabulary를 생성하는 함수

    Args:
        corpus (List[str]): vocabulary를 구성하고자 하는 텍스트 집합
        tokenize_fn (Callable): 토크나이징을 수행할 수 있는 함수
    Returns:
        List[str]: sorted unique vocabulary
    """
    vocab = []
    for text in corpus:
        tokenized_text = tokenize_fn(text)
        vocab += tokenized_text
    return sorted(set(vocab))

vocab = get_vocab(["i love you", "good morning", "i like you"], word_tokenize)
print(f"{vocab=}")

vocab=[' ', 'good', 'i', 'like', 'love', 'morning', 'you']


In [27]:
def token_to_id(vocab: List[str]) -> Dict[str, int]:
    """
    Vocabulary를 입력받아 token -> id 딕셔너리를 반환하는 함수
    """
    return {t: i for i, t in enumerate(vocab)}

def id_to_token(vocab: List[str]) -> Dict[int, str]:
    """
    Vocabulary를 입력받아 id -> token 딕셔너리를 반환하는 함수
    """
    return {i: t for i, t in enumerate(vocab)}

print(f"{token_to_id(vocab)=}")
print(f"{id_to_token(vocab)=}")

token_to_id(vocab)={' ': 0, 'good': 1, 'i': 2, 'like': 3, 'love': 4, 'morning': 5, 'you': 6}
id_to_token(vocab)={0: ' ', 1: 'good', 2: 'i', 3: 'like', 4: 'love', 5: 'morning', 6: 'you'}


In [28]:
def encoder(tokens: List[str], token2id: Dict[str, int]) -> List[int]:
    token_ids = [token2id[token] for token in tokens]
    return token_ids

def decoder(token_ids: List[int], id2token: Dict[int, str]) -> List[str]:
    tokens = [id2token[token_id] for token_id in token_ids]
    return tokens

token2id, id2token = token_to_id(vocab), id_to_token(vocab)
encoded_text = encoder(word_tokenize("i love you"), token2id)
decoded_text = decoder(encoded_text, id2token)

print(f"{encoded_text=}")
print(f"{decoded_text=}")

encoded_text=[2, 0, 4, 0, 6]
decoded_text=['i', ' ', 'love', ' ', 'you']


### 1.4 공용 Util & Baseline
실험 전반에서 재사용할 Utility 함수 정의

- **`train_step` / `evaluate`** — 학습 루프와 held-out 평가
    - 평가 metric은 실험마다 다르므로 **주입 가능하게** (예: next-token acc, recall-위치 acc)
- **n-gram Baseline** — 학습 코퍼스에서 (n-1)-gram → 다음 토큰 **카운트 테이블**
    - `unigram` (맥락 0, floor = 토큰 분포 엔트로피)과 `bigram` (직전 1토큰) 준비
    - 목적: RNN의 loss/accuracy를 **"무엇보다 좋은가"** 로 해석 가능하게 만드는 기준선

In [31]:
def ids_to_onehot(token_ids: List[int], vocab_size: int) -> torch.FloatTensor:
    """
    토큰들의 id 시퀀스를 one-hot encoding하여 하나의 FloatTensor로 변환하는 함수

    Args:
        token_ids (List[int]): 특정 문자열의 token id 시퀀스
        vocab_size (int): vocabulary의 크기
    Returns:
        (torch.FloatTensor): 시퀀스 하나의 embedding 텐서
    """
    return F.one_hot(
        torch.LongTensor(token_ids),
        num_classes=vocab_size
    ).float()

print(ids_to_onehot([0, 1, 2, 3, 4], vocab_size=5))

tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])


In [ ]:
def train_step(model: nn.Module, X, y, loss_fn, optimizer, scheduler=None, device="cpu"):
    # 1. 데이터를 device로 이동
    X, y = X.to(device), y.to(device)
    # 2. 이전 step에서의 grad 초기화
    optimizer.zero_grad()
    # 3. 현재 step에서의 loss 계산 (forward)
    logits, _ = model(X) # logit.shape = (num_layers, seq_len, vocab_size)
    vocab_size = logits.size(-1)

    logits = logits.reshape(-1, vocab_size)
    y = y.reshape(-1)

    loss = loss_fn(logits, y) # CrossEntropyLoss는 (N, C) logits와 (N,) target을 기대
    
    # 4. Gradient 계산 (backward)
    loss.backward()
    # 5. Weight 업데이트 (optimization)
    optimizer.step()
    # (옵션) 6. Learning Rate 스케쥴링
    if scheduler is not None:
        scheduler.step()

    return loss.item() # .item()으로 loss의 float 값만을 반환    

In [ ]:
def evaluate(
    model: nn.Module,
    dataset,
    metric_fn,
    device = "cpu"
):
    model.eval()
    with torch.no_grad():
        total_numerator, total_denominator = 0, 0

        for X, y in dataset:
            X, y = X.to(device), y.to(device)
            logits, _ = model(X)
            vocab_size = logits.size(-1)

            logits = logits.reshape(-1, vocab_size)
            y = y.reshape(-1)

            numerator, denominator = metric_fn(logits, y)
            total_numerator += numerator
            total_denominator += denominator
        return total_numerator / total_denominator

In [ ]:
def accuracy(pred, true, ignore_index=-100) -> Tuple[int, int]:
    """
    Args:
        pred (torch.Tensor): (N, C, ...)의 shape을 가지는 모델의 예측
        true (torch.Tensor): (N, ...)의 shape을 가지는 정답
    Returns:
        correct (int): pred와 true가 일치하는 개수
        total (int): 전체 데이터 개수
    """
    valid_flags = (true != ignore_index) # ignore_index를 값으로 갖는 위치는 측정하지 않음
    pred_labels = pred.argmax(dim=1)
    correct = ((pred_labels == true) & valid_flags).sum() # valid_flag가 true인 것만 사용
    total = valid_flags.sum()
    return correct.item(), total.item()

def cross_entropy(pred, true, ignore_index=-100) -> Tuple[float, int]:
    """
    Args:
        pred (torch.Tensor): (N, C, ...)의 shape을 가지는 모델의 예측
        true (torch.Tensor): (N, ...)의 shape을 가지는 정답
    Returns:
        total_loss (float): 유효한 토큰들의 Negative log-likelihood 합
        total_tokens (int): 전체 유효한 토큰의 수
    """
    total_loss = F.cross_entropy(pred, true, ignore_index=ignore_index)
    total_tokens = (true != ignore_index).sum()
    return total_loss.item(), total_tokens.item()

## 2. 실험 A — 반복 데이터 (char-level)

### 2.1 명제 & 성공 기준
- **명제**: RNN이 bigram/n-gram 같은 고정 윈도우 모델보다 **긴 맥락을 실제로 활용**한다.
- **데이터**: 특정 패턴이 반복되는 문자열 (예: `"i love you "` × 반복). char-level.
- **성공 기준**: held-out 시퀀스에서 RNN의 next-token accuracy가 **bigram baseline을 유의미하게 상회**.
- **주의**: periodic 문자열은 저차 n-gram도 상당 부분 풂 → RNN이 이겨도 *"조금 더 긴 맥락"* 수준의 **약한 증거**임을 명시.

### 진행 단계
- **2.2** 데이터 생성 + train/eval split
- **2.3** Baseline: bigram/n-gram accuracy
- **2.4** RNN 학습 & held-out accuracy
- **2.5** 결론: RNN vs baseline 격차로 명제 판정

*(아래 코드 셀부터 2.2 시작)*

## 3. 실험 B — Long-range 의존성 (char-level)

### 3.1 명제 & 성공 기준
- **명제**: 간격 N이 커질수록 vanilla RNN의 **장기의존성이 무너진다** (vanishing gradient).
- **데이터**: `첫문자 + filler(N) + 첫문자` — 마지막 문자가 첫 문자를 **결정적으로** 따라감.
    - filler vocab에서 **signal 문자 제외** (정답 유출 방지)
    - **signal 알파벳은 작게 (2~4개)** → chance level 명확
    - **filler 난이도 축**: 전역 고정(best case) ↔ 위치별 랜덤(worst case, 간섭)
- **Metric**: **recall 위치(마지막 문자) accuracy만** 측정, 중간 위치 loss는 **마스킹**
    - ⚠️ 전 위치 평균 loss 금지 — 랜덤 중간이 신호를 익사시킴
- **Baseline**: n-gram은 **구조적으로 불가**(의존성이 윈도우 밖) + **chance level** (= 1/|signal|)
- **성공 기준 (장점)**: 작은 N에서 RNN recall ≫ chance, n-gram 불가
- **한계 관찰 (단점)**: N을 키우면 RNN recall이 chance로 수렴

### 진행 단계
- **3.2** 데이터 생성 (N × filler 난이도)
- **3.3** Baseline & chance level
- **3.4** RNN 학습 — recall 위치 평가
- **3.5** 핵심 그림: **recall accuracy vs N** (filler 유형별 상·하한 밴드)
- **3.6** 결론: 장점(n-gram 불가를 RNN이 품)과 단점(큰 N 붕괴) 동시 판정

## 4. 실험 C — 실제 텍스트 & 생성 동역학 (word-level)

### 4.1 명제 & 성공 기준
- **명제**: Language Model은 **학습(teacher forcing) ≠ 추론(autoregressive)** — 자기 예측을 다시 입력으로 먹으며 오차가 누적된다 (**exposure bias**).
- **역할**: 정성(qualitative) 데모. 정량 "효과성 증명"은 A·B가 담당하므로 여기선 **생성 동역학**에만 집중 (암기 여부는 논점이 아님).
- **데이터**: 짧은 문장 여러 개, word-level.
- **관찰 포인트**:
    - teacher-forcing perplexity (정답을 입력으로 줄 때)
    - 자기회귀 생성 — 길이가 늘수록 문장이 **붕괴하는 지점**

### 진행 단계
- **4.2** 데이터 & 학습
- **4.3** Teacher-forcing perplexity
- **4.4** 자기회귀 생성 & 붕괴 관찰
- **4.5** 결론: 학습·추론 동작 차이 정리

## 5. 종합 결론
학습 목표별로 각 실험이 무엇을 보였는지 회수한다.

- **성능 (A)**: RNN이 고정 윈도우 baseline(bigram)을 넘어 **맥락을 활용**함
- **한계 (B)**: 간격이 멀어지면 **장기의존성이 붕괴** — `recall accuracy vs N` 밴드로 확인
- **학습·추론 차이 (C)**: teacher forcing vs 자기회귀 생성의 **exposure bias**

### 다음 챕터로
B에서 드러난 **장기의존성 한계** → LSTM/GRU의 gating이 이를 어떻게 완화하는지,
이후 **Attention/Transformer** 가 고정 순차 전달의 병목을 어떻게 제거하는지로 연결.